# Sudoku example

Solving Sudokus is a nice toy example with many implementations for different formal verification tools ([1](https://blog.yosyshq.com/p/solving-sudoku-with-sby/), [2](https://github.com/vmunoz82/sudoku-challenge?tab=readme-ov-file), [3](https://de.mathworks.com/company/technical-articles/a-model-checking-example-solving-sudoku-using-simulink-design-verifier.html)...).

The goal is not to verify some HDL design. Instead the tool is used directly to find some state that satisfies all Sudoku rules.

The basic strategy of my approach looks like this:

1. create a 9x9 Anyconst array with one entry for each Sudoku cell
2. each Sudoku cell is represented by a bit mask of all allowed numbers in the cell
3. add assumptions that limit the array state to valid Sudokus
    1. every cell has one concrete value

        ⇒ every cell bitmask is onehot
    2. every row/column/box contains every number
    
        ⇒ or-operation over the elements of every group results in all hot vector
    3. some values are predefined

        ⇒ add them as assumptions
4. use solver to find a solution that satisfies all assumptions

The solution has 81 elements. To make it a bit more readable I added some logic that outputs the result to 9 ports over 9 clock cycles.

In [1]:
# basic setup of jupyter notebook

from example_util.jupyter_util import display_vcd

import cohdl
from cohdl import Bit, Unsigned, BitVector, Full, Port
from cohdl_yosys import YosysParams, YosysTestCase

from cohdl_yosys.formal import (
    wait,
    ticks_since_start,
    set_default_ctx,
    Anyconst,
    assume,
    cover,
    is_onehot
)

import cohdl.std as std

cohdl.use_pretty_traceback(False)

In [2]:
W = H = 3
N = W*H

# entity only used to output result
class SudokuEntity(cohdl.Entity):
    clk = Port.input(Bit)

    row_1 = Port.input(Unsigned[4])
    row_2 = Port.input(Unsigned[4])
    row_3 = Port.input(Unsigned[4])

    row_4 = Port.input(Unsigned[4])
    row_5 = Port.input(Unsigned[4])
    row_6 = Port.input(Unsigned[4])

    row_7 = Port.input(Unsigned[4])
    row_8 = Port.input(Unsigned[4])
    row_9 = Port.input(Unsigned[4])

    def architecture(self):
        pass


In [3]:
@cohdl.pyeval
def sudoku_groups(board):
    rows = [[] for _ in range(N)]
    cols = [[] for _ in range(N)]
    blocks = [[] for _ in range(N)]

    w_blocks = N // W

    for row in range(N):
        for col in range(N):
            entry = board[row][col]

            rows[row].append(entry)
            cols[col].append(entry)
            blocks[(row // H) * w_blocks + col // W].append(entry)

    return [*rows, *cols, *blocks]

# find a solution with diagonal set to 1-2-3-1-2-3-1-2-3
known_fields = [(0, 0, 1), (1, 1, 2), (2, 2, 3),
                (3, 3, 1), (4, 4, 2), (5, 5, 3),
                (6, 6, 1), (7, 7, 2), (8, 8, 3)]

class SudokuSolver(YosysTestCase, entity=SudokuEntity):
    _yosys_params_ = YosysParams(cover=True, cover_depth=N, clean_build_dir=True, quiet=True)

    def architecture(self, dut: SudokuEntity):

        set_default_ctx(std.SequentialContext(std.Clock(dut.clk)))

        board = Anyconst[std.Array[std.Array[BitVector[N], N], N]]()

        @std.concurrent
        def solve_sudoku():

            for row in range(N):
                for col in range(N):
                    assume[:"all_cells_are_onehot"](is_onehot(board[row][col]))

            for group in sudoku_groups(board):
                assume[:"all_groups_are_solved"](
                    std.batched_fold(lambda a, b: a | b, group) == Full
                )

            for row, col, val in known_fields:
                assume[:"known_fields"](board[row][col] == std.one_hot(N, val - 1))

            #
            # cover and display

            cover["display_duration"](wait[N])

            for row in range(N):
                assume[:"output_display_value"](
                    getattr(dut, "row_{}".format(row + 1))
                    == std.count_trailing_zeros(board[row][ticks_since_start() - 1]) + 1
                )

SudokuSolver().test_formal_properties()

display_vcd("build/**/*.vcd", "(*row_)*")

build/project_cover/engine_0/trace0.vcd
